# Train: monument inpainting

Fine-tune Stable Diffusion 2 inpainting with LoRA on the Indian monuments dataset.

Run the cells from top to bottom. All settings are in the first code cell.

## 1. Settings

In [ ]:
BASE_MODEL = "sd2-community/stable-diffusion-2-inpainting"
TRAIN_LIST = "data/splits/monuments_train.txt"    # one image path on each line
VAL_LIST = "data/splits/monuments_val.txt"
OUTPUT_DIR = "runs/monuments"

IMAGE_SIZE = 512         # 256 is faster
MAX_TRAIN = None         # None = all images. For a quick test, use 40
MAX_VAL = 200            # for a quick test, use 8
EPOCHS = 6               # for a quick test, use 1
BATCH_SIZE = 1
GRAD_ACCUM = 1
LR = 5e-6
LR_SCHEDULER = "constant"
WARMUP_STEPS = 0
AUGMENT = True           # random flip + colour jitter
GRADIENT_CHECKPOINTING = True

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

PROMPT = ""              # empty prompt during training
SEED = 42

## 2. Imports and device

In [ ]:
import os
os.environ.setdefault("HF_HOME", os.path.abspath(".hf_cache"))   # model weights are kept in this folder

import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from diffusers.optimization import get_scheduler
from peft import LoraConfig, get_peft_model
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import CLIPTextModel, CLIPTokenizer

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"          # mixed precision (fp16) only on NVIDIA GPUs
print("device:", DEVICE)

## 3. Masks and dataset

In [ ]:
def make_mask(size, rng=random):
    """One random rectangle. Each side is between size/10 and size/3. White (255) = hole."""
    mask = Image.new("L", (size, size), 0)
    draw = ImageDraw.Draw(mask)
    w = rng.randint(size // 10, size // 3)
    h = rng.randint(size // 10, size // 3)
    x = rng.randint(0, size - w)
    y = rng.randint(0, size - h)
    draw.rectangle([x, y, x + w, y + h], fill=255)
    return mask

In [ ]:
def read_list(path, limit=None):
    lines = [l.strip() for l in open(path) if l.strip()]
    return lines[:limit] if limit else lines


class InpaintDataset(Dataset):
    def __init__(self, paths, augment=False, fixed_masks=False):
        self.paths = paths
        self.augment = augment
        self.fixed_masks = fixed_masks      # validation: the same mask for an image each time
        self.aug = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.1, 0.1)])
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC)
        if self.augment:
            img = self.aug(img)             # augment before masking, so image and masked image agree
        rng = random.Random(SEED + i) if self.fixed_masks else random
        mask = (self.to_tensor(make_mask(IMAGE_SIZE, rng)) > 0.5).float()   # 1 = hole
        image = self.to_tensor(img) * 2 - 1                                 # range [-1, 1]
        masked = image * (1 - mask)                                         # hole filled with 0 (grey)
        return {"image": image, "mask": mask, "masked": masked}


train_ds = InpaintDataset(read_list(TRAIN_LIST, MAX_TRAIN), augment=AUGMENT)
val_ds = InpaintDataset(read_list(VAL_LIST, MAX_VAL), fixed_masks=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=1)
print("train images:", len(train_ds), " validation images:", len(val_ds))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k in range(4):
    s = train_ds[k]
    axes[0, k].imshow((s["image"].permute(1, 2, 0) + 1) / 2)
    axes[1, k].imshow((s["masked"].permute(1, 2, 0) + 1) / 2)
    axes[0, k].axis("off")
    axes[1, k].axis("off")
axes[0, 0].set_title("image")
axes[1, 0].set_title("masked image")
plt.show()

## 4. Models

The VAE and the UNet are frozen. Only the small LoRA layers in the UNet attention are trained.

In [ ]:
# The text prompt is the same for all images, so encode it one time and delete the text encoder.
tokenizer = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(BASE_MODEL, subfolder="text_encoder", variant="fp16",
                                             torch_dtype=torch.float32).to(DEVICE)
ids = tokenizer(PROMPT, padding="max_length", max_length=tokenizer.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad():
    text_emb = text_encoder(ids)[0]
del text_encoder

vae = AutoencoderKL.from_pretrained(BASE_MODEL, subfolder="vae", variant="fp16",
                                    torch_dtype=torch.float32).to(DEVICE)
vae.requires_grad_(False)
noise_sched = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")

unet = UNet2DConditionModel.from_pretrained(BASE_MODEL, subfolder="unet", variant="fp16",
                                            torch_dtype=torch.float32)
unet.requires_grad_(False)
if GRADIENT_CHECKPOINTING:
    unet.enable_gradient_checkpointing()    # uses less memory, a little slower
unet = get_peft_model(unet, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                       target_modules=["to_q", "to_k", "to_v", "to_out.0"], bias="none"))
unet.to(DEVICE)
unet.print_trainable_parameters()

## 5. Loss

In [ ]:
def get_loss(batch):
    image = batch["image"].to(DEVICE)
    mask = batch["mask"].to(DEVICE)
    masked = batch["masked"].to(DEVICE)

    with torch.no_grad():
        latents = vae.encode(image).latent_dist.sample() * vae.config.scaling_factor
        masked_latents = vae.encode(masked).latent_dist.sample() * vae.config.scaling_factor

    noise = torch.randn_like(latents)
    t = torch.randint(0, noise_sched.config.num_train_timesteps, (latents.shape[0],), device=DEVICE)
    noisy_latents = noise_sched.add_noise(latents, noise, t)
    small_mask = F.interpolate(mask, size=latents.shape[-2:], mode="nearest")

    # The inpainting UNet takes 9 channels: noisy latents (4) + mask (1) + masked-image latents (4)
    unet_input = torch.cat([noisy_latents, small_mask, masked_latents], dim=1)
    with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
        pred = unet(unet_input, t, encoder_hidden_states=text_emb.expand(len(t), -1, -1)).sample
    return F.mse_loss(pred.float(), noise)      # the model predicts the noise

## 6. Training

In [ ]:
params = [p for p in unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR)
total_steps = EPOCHS * len(train_loader) // GRAD_ACCUM
lr_sched = get_scheduler(LR_SCHEDULER, optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

history = []
best_val = float("inf")

for epoch in range(EPOCHS):
    # ---- train ----
    unet.train()
    total = 0.0
    bar = tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}")
    for i, batch in enumerate(bar):
        loss = get_loss(batch)
        scaler.scale(loss / GRAD_ACCUM).backward()
        total += loss.item()
        if (i + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            lr_sched.step()
        bar.set_postfix(loss=f"{total / (i + 1):.4f}")
    train_loss = total / len(train_loader)

    # ---- validate (same noise and same masks each epoch) ----
    unet.eval()
    rng_state = torch.random.get_rng_state()
    torch.manual_seed(SEED)
    with torch.no_grad():
        val_loss = float(np.mean([get_loss(b).item() for b in tqdm(val_loader, desc="validation", leave=False)]))
    torch.random.set_rng_state(rng_state)

    history.append((train_loss, val_loss))
    print(f"epoch {epoch + 1}: train loss {train_loss:.4f}   validation loss {val_loss:.4f}")

    # ---- save the LoRA adapter (small files, not the full UNet) ----
    unet.save_pretrained(f"{OUTPUT_DIR}/last")
    if val_loss < best_val:
        best_val = val_loss
        unet.save_pretrained(f"{OUTPUT_DIR}/best")
        print("  saved new best model")

print("Done. Best validation loss:", round(best_val, 4))

In [ ]:
plt.plot([h[0] for h in history], label="train")
plt.plot([h[1] for h in history], label="validation")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/loss.png")
plt.show()